# Issue #5: Hyperparameter Sensitivity Analysis (LoRA vs DoRA)

This notebook sweeps over hyperparameter combinations (adapter type, rank, learning rate)
for both LoRA and DoRA, then evaluates each on a subset of commonsense reasoning datasets.

**Goal**: Determine how sensitive each adapter is to hyperparameter choices and whether
DoRA is more robust than LoRA across configurations.

In [ ]:
# Cell 1: Setup
!pip install -q torch transformers datasets peft fire matplotlib numpy pandas

import os
if not os.path.exists('dsa5106-group-2'):
    !git clone https://github.com/jeykori/dsa5106-group-2.git

%cd dsa5106-group-2

if not os.path.exists('datasets/commonsense_170k.json'):
    !bash scripts/init-datasets.sh

import sys
sys.path.insert(0, './reproduction')

print('Setup complete.')

In [ ]:
# Cell 2: Hugging Face login
from huggingface_hub import login
login()

In [ ]:
# Cell 3: Sweep configuration (edit this cell to adjust the sweep)

SWEEP_CONFIGS = [
    {"adapter": "dora", "lora_r": 4,  "lora_alpha": 8,  "learning_rate": 1e-4},
    {"adapter": "dora", "lora_r": 4,  "lora_alpha": 8,  "learning_rate": 2e-4},
    {"adapter": "lora", "lora_r": 4,  "lora_alpha": 8,  "learning_rate": 1e-4},
    {"adapter": "lora", "lora_r": 4,  "lora_alpha": 8,  "learning_rate": 2e-4},
    {"adapter": "dora", "lora_r": 16, "lora_alpha": 32, "learning_rate": 2e-4},
    {"adapter": "lora", "lora_r": 16, "lora_alpha": 32, "learning_rate": 2e-4},
]

EVAL_DATASETS = ["boolq", "piqa", "hellaswag"]

# Colab-friendly settings (reduce for faster iteration)
SAMPLE_SIZE = 2000
MICRO_BATCH_SIZE = 4
BATCH_SIZE = 16
NUM_EPOCHS = 3
EVAL_STEPS = 40
SAVE_STEPS = 40

OUTPUT_BASE = "./sweep_results"

def config_name(cfg):
    return f"{cfg['adapter']}-r{cfg['lora_r']}-lr{cfg['learning_rate']}"

print(f"Sweep: {len(SWEEP_CONFIGS)} configs x {len(EVAL_DATASETS)} datasets")
for cfg in SWEEP_CONFIGS:
    print(f"  {config_name(cfg)}")

In [ ]:
# Cell 4: Training function

import json
import torch
import transformers
from datasets import load_dataset
from lora import inject_lora
from dora import inject_dora, merge_and_unload_dora
from utils import generate_prompt


def tokenize_prompt(data, tokenizer):
    user_prompt = generate_prompt({**data, "output": ""})
    full_prompt = generate_prompt(data) + tokenizer.eos_token
    tokenized_user_prompt = tokenizer(user_prompt, padding=False)
    tokenized = tokenizer(full_prompt, padding=False)
    user_prompt_len = len(tokenized_user_prompt["input_ids"])
    labels = tokenized["input_ids"].copy()
    labels = [-100] * user_prompt_len + labels[user_prompt_len:]
    tokenized["labels"] = labels
    return tokenized


def run_single_experiment(cfg, model_name="unsloth/llama-3.2-3b"):
    name = config_name(cfg)
    output_dir = f"{OUTPUT_BASE}/{name}/model"
    os.makedirs(output_dir, exist_ok=True)

    target_modules = ["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"]
    gradient_accumulation_steps = BATCH_SIZE // MICRO_BATCH_SIZE

    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", dtype=torch.bfloat16
    )

    if cfg["adapter"] == "lora":
        model = inject_lora(model, r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                            lora_dropout=0.05, target_modules=target_modules)
    elif cfg["adapter"] == "dora":
        model = inject_dora(model, r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                            lora_dropout=0.05, target_modules=target_modules)

    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"

    data = load_dataset("json", data_files="./datasets/commonsense_170k.json")
    data["train"] = data["train"].shuffle(seed=42).select(range(SAMPLE_SIZE))
    data_split = data["train"].train_test_split(test_size=120, shuffle=True, seed=42)
    data_train = data_split["train"].shuffle().map(lambda x: tokenize_prompt(x, tokenizer))
    data_val = data_split["test"].shuffle().map(lambda x: tokenize_prompt(x, tokenizer))

    trainer = transformers.Trainer(
        model=model,
        train_dataset=data_train,
        eval_dataset=data_val,
        args=transformers.TrainingArguments(
            output_dir=output_dir,
            gradient_accumulation_steps=gradient_accumulation_steps,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=cfg["learning_rate"],
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            per_device_train_batch_size=MICRO_BATCH_SIZE,
            bf16=True,
            save_total_limit=1,
            gradient_checkpointing=True,
            optim="adamw_torch",
            warmup_steps=100,
        ),
        data_collator=transformers.DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable} | Total: {total} | %: {100 * trainable / total:.4f}")

    trainer.train()

    # Save training loss history
    log_history = trainer.state.log_history
    with open(f"{OUTPUT_BASE}/{name}/log_history.json", "w") as f:
        json.dump(log_history, f, indent=2)

    if cfg["adapter"] == "dora":
        merge_and_unload_dora(model)

    trainer.save_model(output_dir)
    print(f"Model saved to {output_dir}")

    # Free GPU memory
    del model, trainer
    torch.cuda.empty_cache()

    return output_dir


print('run_single_experiment() defined.')

In [ ]:
# Cell 5: Run training loop (saves progress after each config)

import json

progress_file = f"{OUTPUT_BASE}/sweep_progress.json"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# Resume from previous progress if available
if os.path.exists(progress_file):
    with open(progress_file) as f:
        progress = json.load(f)
    print(f"Resuming: {len(progress)} configs already completed")
else:
    progress = {}

for cfg in SWEEP_CONFIGS:
    name = config_name(cfg)
    if name in progress:
        print(f"Skipping {name} (already completed)")
        continue

    model_dir = run_single_experiment(cfg)
    progress[name] = {"model_dir": model_dir, "config": cfg}

    with open(progress_file, "w") as f:
        json.dump(progress, f, indent=2, default=str)

print(f"\nAll {len(SWEEP_CONFIGS)} training runs complete.")

In [ ]:
# Cell 6: Evaluation loop (reuses reproduction/evaluate.py)

import subprocess

for name, info in progress.items():
    model_dir = info["model_dir"]
    eval_dir = f"{OUTPUT_BASE}/{name}/eval"
    os.makedirs(eval_dir, exist_ok=True)

    for dataset in EVAL_DATASETS:
        outfile = f"{eval_dir}/{dataset}_results.json"
        if os.path.exists(outfile):
            print(f"Skipping {name}/{dataset} (already evaluated)")
            continue

        print(f"Evaluating {name} on {dataset}...")
        result = subprocess.run(
            [
                sys.executable, "reproduction/evaluate.py",
                "--model_path", model_dir,
                "--dataset", dataset,
                "--outfile", outfile,
                "--batch_size", "8",
            ],
            capture_output=True, text=True
        )
        print(result.stdout[-200:] if result.stdout else result.stderr[-200:])

print("\nAll evaluations complete.")

In [ ]:
# Cell 7: Aggregate results into a summary table

import json
import pandas as pd

rows = []
for name, info in progress.items():
    row = {"config": name, "adapter": info["config"]["adapter"],
           "rank": info["config"]["lora_r"], "lr": info["config"]["learning_rate"]}

    for dataset in EVAL_DATASETS:
        outfile = f"{OUTPUT_BASE}/{name}/eval/{dataset}_results.json"
        if os.path.exists(outfile):
            with open(outfile) as f:
                results = json.load(f)
            passed = sum(1 for r in results if r["passed"])
            total = len(results)
            row[dataset] = round(100 * passed / total, 2) if total > 0 else None
        else:
            row[dataset] = None

    dataset_scores = [row[d] for d in EVAL_DATASETS if row.get(d) is not None]
    row["average"] = round(sum(dataset_scores) / len(dataset_scores), 2) if dataset_scores else None
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))
df.to_csv(f"{OUTPUT_BASE}/sweep_summary.csv", index=False)
print(f"\nSaved to {OUTPUT_BASE}/sweep_summary.csv")

In [ ]:
# Cell 8: Grouped bar chart — accuracy by dataset

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(EVAL_DATASETS))
n_configs = len(df)
width = 0.8 / n_configs

colors = plt.cm.Set2(np.linspace(0, 1, n_configs))

for i, (_, row) in enumerate(df.iterrows()):
    values = [row.get(d, 0) or 0 for d in EVAL_DATASETS]
    ax.bar(x + i * width - 0.4 + width/2, values, width, label=row['config'], color=colors[i])

ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy by Dataset and Configuration')
ax.set_xticks(x)
ax.set_xticklabels(EVAL_DATASETS)
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(f'{OUTPUT_BASE}/accuracy_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9: Learning rate sensitivity — accuracy vs LR for each adapter+rank

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, rank in zip(axes, df['rank'].unique()):
    subset = df[df['rank'] == rank]
    for adapter in ['dora', 'lora']:
        adapter_data = subset[subset['adapter'] == adapter].sort_values('lr')
        if len(adapter_data) > 0:
            style = '-o' if adapter == 'dora' else '--s'
            color = '#2ca02c' if adapter == 'dora' else '#1f77b4'
            ax.plot(adapter_data['lr'], adapter_data['average'], style,
                    label=adapter.upper(), color=color, markersize=8)

    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Average Accuracy (%)')
    ax.set_title(f'Rank = {rank}')
    ax.legend()
    ax.set_xscale('log')

fig.suptitle('Learning Rate Sensitivity', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_BASE}/lr_sensitivity.png', dpi=150)
plt.show()

In [ ]:
# Cell 10: Training loss curves

fig, ax = plt.subplots(figsize=(12, 6))

colors_map = plt.cm.Set2(np.linspace(0, 1, len(progress)))

for i, (name, info) in enumerate(progress.items()):
    log_file = f"{OUTPUT_BASE}/{name}/log_history.json"
    if not os.path.exists(log_file):
        continue
    with open(log_file) as f:
        log_history = json.load(f)

    steps = [entry['step'] for entry in log_history if 'loss' in entry]
    losses = [entry['loss'] for entry in log_history if 'loss' in entry]

    if steps:
        style = '-' if info['config']['adapter'] == 'dora' else '--'
        ax.plot(steps, losses, style, label=name, color=colors_map[i], linewidth=1.5)

ax.set_xlabel('Step')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Curves')
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(f'{OUTPUT_BASE}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()